In [1]:
!pip -q install transformers datasets accelerate

In [2]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling,
                          TrainingArguments, Trainer, set_seed)
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [3]:
# Custom dataset (replace with your own text: lines = open("mytext.txt").read().split("\n"))
lines = [
    "Machine learning helps students analyse data and build smart applications.",
    "In our college lab, students train neural networks on small datasets every week.",
    "Natural language processing allows computers to read, understand and write human language.",
    "A good dataset is more important than a complex model.",
    "Students present their projects at the annual technical festival of the college.",
    "Python is the favourite programming language of data science students.",
    "The professor explained how transformers use attention to understand sentences.",
    "Before training a model we clean the data and remove missing values.",
    "Deep learning models need a lot of data and powerful GPUs.",
    "The library stays open late during exam weeks so students can study together.",
    "A chatbot can answer simple questions of students about courses and schedules.",
    "Teamwork and regular practice make every student a better programmer.",
    "Cloud platforms like Google Colab give free GPUs to learners.",
    "Language models predict the next word by learning patterns from large text collections.",
    "Every semester the department organises workshops on artificial intelligence.",
] * 4
dataset = Dataset.from_dict({"text": lines})
print(len(dataset), "training lines")

60 training lines


In [4]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

def generate(prompt, n=2, max_new_tokens=40):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**enc, do_sample=True, top_k=50, top_p=0.95, temperature=0.8,
                             max_new_tokens=max_new_tokens, num_return_sequences=n,
                             pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o, skip_special_tokens=True) for o in out]

prompt = "In our college lab, students"
print("===== BEFORE fine-tuning =====")
for t in generate(prompt): print("-", t.replace("\n", " "))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

===== BEFORE fine-tuning =====
- In our college lab, students study the role of DNA in the development and maintenance of healthy chromosomes, and we are not immune to genetic abnormalities. In the field of genetics, we are concerned with the role of DNA in the maintenance
- In our college lab, students were given the option of spending three hours a day in a designated space, a room or a space shared by a room of their choice. In one test, participants were asked to write a letter in


In [5]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64)

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="gpt2_custom",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=20,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)
trainer.train()

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
20,3.721089
40,1.347246
60,0.520308
80,0.342442


TrainOutput(global_step=90, training_loss=1.3407645675871107, metrics={'train_runtime': 11.1288, 'train_samples_per_second': 32.349, 'train_steps_per_second': 8.087, 'total_flos': 2578217472000.0, 'train_loss': 1.3407645675871107, 'epoch': 6.0})

In [6]:
print("===== AFTER fine-tuning =====")
for p in ["In our college lab, students", "Natural language processing", "A chatbot can"]:
    print(f"\nPrompt: {p}")
    for t in generate(p): print("-", t.replace("\n", " "))

===== AFTER fine-tuning =====

Prompt: In our college lab, students
- In our college lab, students train neural networks on small datasets every week. Students train neural networks on small datasets every week. Our students train neural networks on large datasets every week. Every week, tens of thousands of students build and
- In our college lab, students train neural networks on small datasets every week. Students train neural networks on large datasets every week. Each week, students present their work at the annual technical festival of the college.  Students train neural

Prompt: Natural language processing
- Natural language processing allows computers to read, understand and write human language.  Students train neural networks on small datasets every week. Students then use these training datasets to build smart applications.  In December 2016,
- Natural language processing allows computers to read, understand and write human language. This allows computers to read, write and 

In [7]:
# Effect of decoding settings
p = "Machine learning helps"
enc = tokenizer(p, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    greedy = model.generate(**enc, max_new_tokens=30, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    beam = model.generate(**enc, max_new_tokens=30, num_beams=5, no_repeat_ngram_size=2, early_stopping=True,
                          pad_token_id=tokenizer.eos_token_id)
print("Greedy:", tokenizer.decode(greedy[0], skip_special_tokens=True))
print("Beam  :", tokenizer.decode(beam[0], skip_special_tokens=True))
model.save_pretrained("gpt2_custom_model"); tokenizer.save_pretrained("gpt2_custom_model")

Greedy: Machine learning helps students analyse data and build smart applications.

Machine learning helps students analyse data and build smart applications. Machine learning helps students analyse data and build smart
Beam  : Machine learning helps students analyse data and build smart applications. Students train neural networks on small datasets every week.

The department organises workshops on artificial intelligence and artificial


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('gpt2_custom_model/tokenizer_config.json', 'gpt2_custom_model/tokenizer.json')